### Small-Scale (Gene) Mutationssd

Gene mutations affect individual DNA nucleotides or base pairs: 

1. Substitution (Point Mutation): One base pair is replaced by another. Subtypes include:
  - Silent (Synonymous) Mutation: Changes a nucleotide, but the new codon codes for the exact same amino acid, causing no change to the final protein.
  - Missense Mutation: Changes a nucleotide so that a different amino acid is incorporated into the protein, which can alter or destroy protein function (e.g., sickle cell anemia).
  - Nonsense Mutation: Converts a regular amino acid codon into a premature stop codon, resulting in an incomplete, usually non-functional protein. 
  - SNP:
    - germline
    - somatic

2. InDel
  - Insertion: The addition of one or more extra nucleotides into the DNA sequence. 
  - Deletion: The removal of one or more nucleotides from the DNA sequence. 
3. Frameshift Mutation: Caused by insertions or deletions that are not in multiples of three, shifting the reading frame of the entire genetic message downstream and severely altering the resulting protein. 
4. Multi-Nucleotide Variants (MNV): Two or more neighboring nucleotide changes (often within 2 base pairs or the same protein codon) occurring on the same chromosomal haplotype. Causes: Can arise from independent single-nucleotide changes over time, error-prone DNA replication via polymerase zeta, or polymerase slippage at repeat junctions.

### Large-Scale (Chromosomal) Mutations

Chromosomal mutations alter the structure of a chromosome or the total number of chromosomes: 

1. Duplication: A segment of a chromosome is copied and repeated, leading to extra genetic material.
2. Inversion: A segment of a chromosome is broken off, flipped 180 degrees, and reinserted backwards.
3. Translocation: A piece of one chromosome breaks off and attaches to a completely different, non-homologous chromosome.
4. Aneuploidy (Numerical): An abnormal number of chromosomes, such as having an extra chromosome or missing one, often caused by non-disjunction during cell division. 
5. Polyploidy
5. Mismatch repair
6. Repeat expansion
7. Structural variants

### Mitochondria

1. Mitochondrial heteroplasmy is the presence of more than one type of mitochondrial DNA (mtDNA) within a single cell, tissue, or organism. Cells contain multiple mitochondria with hundreds to thousands of DNA copies; when a mutation arises in some, normal (wild-type) and mutated genomes coexist.

In [1]:
from platform import python_version
print(python_version())

3.10.19


In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

from scipy.stats import spearmanr

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/mutation_analysis")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, str(ROOT_SRC))

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir

root_figure = create_dir(ROOT0, "pictures")
root_results = create_dir(ROOT0, "results")
root_data = create_dir(ROOT0, "data")
root_vcf = create_dir(root_data, "vcf")

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/mutation_analysis
ROOT_SRC added: /home/flavio/uv/mutation_analysis/src


In [3]:
[x for x in os.listdir(root_vcf) if x.endswith('.vcf')]

['10_polyploidy_genotypes.vcf',
 '09_aneuploidy_chromosome_arm.vcf',
 '07_structural_variants.vcf',
 '04_small_indels.vcf',
 '05_mismatch_repair_MSI_homopolymer.vcf',
 '08_cnv_focal_segments.vcf',
 '06_str_repeat_expansion.vcf',
 '01_snp_germline_trio.vcf',
 '03_mnv_dinucleotide.vcf',
 '02_snv_somatic_tumor_normal.vcf',
 '11_mito_heteroplasmy.vcf']

### VCF analyser

In [4]:
from libs.vcf_analyzer import *

In [5]:
PLOIDY_NAME

{0: 'missing',
 1: 'haploid',
 2: 'diploid',
 3: 'triploid',
 4: 'tetraploid',
 5: 'pentaploid',
 6: 'hexaploid'}

In [6]:
files = [x for x in os.listdir(root_vcf) if x.endswith('.vcf.gz')]
files.sort()
files

['01_snp_germline_trio.vcf.gz',
 '02_snv_somatic_tumor_normal.vcf.gz',
 '03_mnv_dinucleotide.vcf.gz',
 '04_small_indels.vcf.gz',
 '05_mismatch_repair_MSI_homopolymer.vcf.gz',
 '06_str_repeat_expansion.vcf.gz',
 '07_structural_variants.vcf.gz',
 '08_cnv_focal_segments.vcf.gz',
 '09_aneuploidy_chromosome_arm.vcf.gz',
 '10_polyploidy_genotypes.vcf.gz',
 '11_mito_heteroplasmy.vcf.gz']

In [7]:
from types import SimpleNamespace

i = 1
VCFS             = [root_vcf / files[i]]  # VCF / VCF.gz / BCF files
OUTDIR           = "vcf_analysis"
BCFTOOLS         = "bcftools"      # bcftools executable (or full path, e.g. /usr/local/bin/bcftools)
REF              = None            # indexed reference FASTA, e.g. "/data/refs/hg38.fa"
INCLUDE          = None            # bcftools filter expression, e.g. "QUAL>30 && FMT/DP>10"
REGIONS          = None            # e.g. "chr17:7661779-7687538"
PASS_ONLY        = False           # keep only FILTER=PASS (or '.') records
BASELINE_PLOIDY  = 2               # expected copy number (4 after genome doubling)
MIN_VAF          = 0.05            # minimum tumor VAF for a somatic call
MAX_NORMAL_VAF   = 0.02            # maximum normal VAF for a somatic call
THREADS          = 2
NO_PLOTS         = False

print(VCFS, VCFS[0].exists())

[PosixPath('/home/flavio/uv/mutation_analysis/data/vcf/02_snv_somatic_tumor_normal.vcf.gz')] True


### Running

```Python
def run(vcfs:list, outdir:Path, bcftools:str, ref:Path|str|None, include:str|None,
        regions:Path|str|None=None, pass_only:bool=False, baseline_ploidy:int=2,
        min_vaf:float=0.05, max_normal_vaf:float=0.02, threads:int=2, no_plots:bool=False):
```

In [8]:
res = run(vcfs=VCFS, outdir=root_results, pass_only=True, include="FMT/DP>=20",
          baseline_ploidy=4, bcftools="/usr/local/bin/bcftools")

# results come back as Python objects, not just files
res[0]["summary"]                 # dict: classes, ploidy, aneuploidy, somatic counts


{'file': '/home/flavio/uv/mutation_analysis/data/vcf/02_snv_somatic_tumor_normal.vcf.gz',
 'fileformat': 'VCFv4.2',
 'samples': ['NORMAL', 'TUMOR'],
 'tumor_normal': ['TUMOR', 'NORMAL'],
 'records_after_filters': 3,
 'variant_classes': {'SNV': 3},
 'event_scales': {},
 'msi_slippage_indels': 0,
 'repeat_expansions': 0,
 'ts_tv': 2.0,
 'bcftools_SN': {'samples': 2,
  'records': 3,
  'no-ALTs': 0,
  'SNPs': 3,
  'MNPs': 0,
  'indels': 0,
  'others': 0,
  'multiallelic sites': 0,
  'multiallelic SNP sites': 0},
 'dominant_ploidy': {'NORMAL': 'diploid', 'TUMOR': 'diploid'},
 'aneuploid_chromosomes': [],
 'arm_level_events': [],
 'somatic_calls': 3}

In [12]:
for key, val in res[0].items():
    print(key)

summary
sites
cn
contigs


In [9]:
dfr = res[0]["sites"]
print(dfr.shape)
dfr

(3, 14)


,VID,CHROM,POS,ID,REF,ALT,QUAL,FILTER,GENE,SOMATIC,CLASS,SIZE,SCALE,REPEAT
0,0,chr7,140753336,.,A,T,.,PASS,BRAF,1,SNV,1,SMALL,
1,1,chr12,25245350,.,C,T,.,PASS,KRAS,1,SNV,1,SMALL,
2,2,chr17,7675088,.,C,T,.,PASS,TP53,1,SNV,1,SMALL,


In [10]:
dfcn = res[0]["cn"]
dfcn

""


In [14]:
dfctg = res[0]["contigs"]
print(type(dfctg))
dfctg

<class 'dict'>


{'chr7': 159345973, 'chr12': 133275309, 'chr17': 83257441}